**Eigenvalues and eigenvectors.**
A nonzero vector $\mathbf{v}$ is an **eigenvector** of a square matrix $\mathbf{A}$
if multiplying by $\mathbf{A}$ only scales it - it does not change direction:

$$\mathbf{A}\cdot\mathbf{v} = \lambda\,\mathbf{v}$$

The scalar $\lambda$ is the corresponding **eigenvalue**.
This notebook demonstrates six key results:

1. The eigenequation $\mathbf{A}\mathbf{v} = \lambda\mathbf{v}$ holds numerically
2. Hermitian matrices have **real** eigenvalues
3. Eigenvectors of a Hermitian matrix are **pairwise orthogonal**
4. **Eigendecomposition**: the eigenbasis diagonalizes a matrix
5. Eigendecomposition enables efficient **matrix exponentiation**
6. Matrices sharing an eigenbasis **commute** under multiplication
7. Eigenvalues and determinant of a **unitary** matrix all have modulus 1

In [ ]:
"""eigenvalues.ipynb"""

# Cell 01 - Create two matrices: A (general complex) and B (Hermitian)

import numpy as np
from IPython.display import Math, display

from qis101_utils import as_latex

a = np.array(
    [
        [-5.664 - 3.623j, 7.672 - 4.470j, 1.864 - 7.149j],
        [0.766 - 4.821j, -4.410 - 0.228j, 9.759 + 4.256j],
        [1.0335 - 3.672j, 3.890 - 5.741j, 7.760 + 3.812j],
    ]
)

b = np.array([[5, 4 + 5j, 6 - 16j], [4 - 5j, 13, 7], [6 + 16j, 7, -2.1]])  # Hermitian

display(as_latex(a, prefix=r"\mathbf{A}="))
display(as_latex(b, prefix=r"\mathbf{B}="))

---
**The eigenequation.**
`np.linalg.eig` returns all eigenvalues and their corresponding eigenvectors.
The eigenvectors are stored as **columns** of the returned matrix, so
eigenvector $k$ is `eigen_vecs[:, k]`.
The cell verifies that $\mathbf{A}\cdot\mathbf{v}_2 = \lambda_2\cdot\mathbf{v}_2$
holds numerically for the second eigenpair.

In [ ]:
# Cell 02 - Verify the eigenequation A*v2 = lambda2 * v2

display(as_latex(a, prefix=r"\mathbf{A}="))

# eig returns eigenvectors as columns of eigen_vecs
eigen_vals, eigen_vecs = np.linalg.eig(a)

display(
    as_latex(eigen_vecs, prefix=r"\mathrm{Eigenvectors\;(\mathbf{v})\;of\;\mathbf{A}}=")
)
display(
    as_latex(eigen_vals, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;\mathbf{A}}=")
)

display(as_latex(eigen_vecs[:, 1], prefix=r"\mathbf{v_2}="))
display(as_latex(eigen_vals[1:2], prefix=r"\mathrm{\lambda_2}="))

t1 = np.dot(a, eigen_vecs[:, 1])
t2 = np.dot(eigen_vals[1], eigen_vecs[:, 1])

display(as_latex(t1, prefix=r"\mathbf{A\cdot v_2}="))
display(as_latex(t2, prefix=r"\mathrm{\lambda_2\cdot\mathbf{v_2}}="))
display(
    Math(
        r"\mathbf{A\cdot v_2}="
        r"\mathrm{\lambda_2\cdot\mathbf{v_2}}"
        rf"\;?\;\rightarrow\;{np.isclose(t1, t2).all()}"
    )
)

---
**Hermitian matrices have real eigenvalues.**
For a Hermitian matrix $\mathbf{B}$ ($\mathbf{B}^\dagger = \mathbf{B}$),
the eigenvalue equation $\mathbf{B}\mathbf{v} = \lambda\mathbf{v}$ implies
$\lambda = \langle\mathbf{v},\mathbf{B}\mathbf{v}\rangle / \langle\mathbf{v},\mathbf{v}\rangle$,
which equals its own conjugate and is therefore real.
`np.linalg.eigh` (rather than `eig`) is used for *Hermitian* matrices because
it exploits the inherent symmetry inside Hermitian matrices, for greater numerical stability and guarantees
real eigenvalues in ascending order.

In [ ]:
# Cell 03 - Hermitian matrices have real eigenvalues (use eigh, not eig)

display(as_latex(b, prefix=r"\mathbf{B}="))

eigen_vals, eigen_vecs = np.linalg.eigh(b)

display(
    as_latex(eigen_vecs, prefix=r"\mathrm{Eigenvectors\;(\mathbf{v})\;of\;}\mathbf{B}=")
)
display(
    as_latex(eigen_vals, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;}\mathbf{B}=")
)

---
**Eigenvectors of a Hermitian matrix are pairwise orthogonal.**
The **spectral theorem** guarantees that a Hermitian matrix has an orthonormal
eigenbasis: its eigenvectors are mutually perpendicular and each has unit norm.
The inner products $\langle\mathbf{v}_i, \mathbf{v}_j\rangle = 0$ for $i \neq j$
are verified here using `np.vdot`, which automatically conjugates its first argument.
The `+ 0` trick converts display of $-0.0$ to $0.0$.

In [ ]:
# Cell 04 - Eigenvectors of a Hermitian matrix are pairwise orthogonal

display(as_latex(b, prefix=r"\mathbf{B}="))
display(as_latex(eigen_vecs[:, 0], prefix=r"\mathbf{v_1}="))
display(as_latex(eigen_vecs[:, 1], prefix=r"\mathbf{v_2}="))
display(as_latex(eigen_vecs[:, 2], prefix=r"\mathbf{v_3}="))

v1_dot_v2 = np.vdot(eigen_vecs[:, 0], eigen_vecs[:, 1])
v2_dot_v3 = np.vdot(eigen_vecs[:, 1], eigen_vecs[:, 2])
v3_dot_v1 = np.vdot(eigen_vecs[:, 2], eigen_vecs[:, 0])

# + 0 converts -0.0 to 0.0 for cleaner display
display(
    Math(rf"\langle\mathbf{{v_1,v_2}}\rangle={np.round(v1_dot_v2, 5) + 0}"),
    Math(rf"\langle\mathbf{{v_2,v_3}}\rangle={np.round(v2_dot_v3, 5) + 0}"),
    Math(rf"\langle\mathbf{{v_3,v_1}}\rangle={np.round(v3_dot_v1, 5) + 0}"),
)

---
**Eigendecomposition and diagonalization.**
A diagonalizable matrix $\mathbf{A}$ can be written as
$\mathbf{A} = \mathfrak{B}\,\mathbf{\Lambda}\,\mathfrak{B}^{-1}$,
where $\mathfrak{B}$ is the **eigenbasis** (matrix of eigenvectors as columns)
and $\mathbf{\Lambda}$ is the diagonal matrix of eigenvalues.

Rearranging: $\mathfrak{B}^{-1}\mathbf{A}\,\mathfrak{B} = \mathbf{\Lambda}$,
so the **transition matrix** $\mathfrak{B}^{-1}\mathbf{A}$ applied to the
eigenbasis $\mathfrak{B}$ produces the diagonal eigenvalue matrix $\mathbf{\Lambda}$.

In [ ]:
# Cell 05 - Eigendecomposition: (B^-1 * A) * B = Lambda (diagonal eigenvalue matrix)

a = np.array(
    [
        [-5.96377 + 0.563478j, -4.65625 - 0.657598j, 1.1037 + 4.11872j],
        [6.01795 - 3.17565j, -1.66041 - 9.61168j, 5.88324 + 1.86256j],
        [0.39357 - 2.25232j, -1.66388 - 15.6493j, 8.02418 + 9.0482j],
    ]
)
display(as_latex(a, prefix=r"\mathbf{A}="))

eigen_vals, eigen_vecs = np.linalg.eig(a)
b = eigen_vecs.copy()  # eigenbasis: columns are eigenvectors
display(as_latex(b, prefix=r"\mathrm{Eigenbasis\;(\mathfrak{B})\;of\;}\mathbf{A}="))

# Transition matrix: B^{-1} * A
m = np.dot(np.linalg.inv(b), a)
display(
    as_latex(
        m,
        prefix=r"\mathrm{Transition\;Matrix\;of\;}(\mathbf{\mathfrak{B}\leftarrow A})=",
    )
)

# (B^{-1} A) * B = Lambda
t1 = np.dot(m, b)
t2 = eigen_vals[np.newaxis] * np.identity(3)

display(
    as_latex(
        t1,
        places=3,
        prefix=(
            r"\mathrm{Diagonal\;Matrix\;(\mathbf{\color{red}{\Lambda}})\;=\;}"
            r"\mathbf{(\mathfrak{B}\leftarrow A)\cdot \mathfrak{B}}="
        ),
    )
)
display(as_latex(t2, places=3, prefix=r"\mathbf{\lambda\;*\;\;I}="))
display(
    Math(
        r"\mathbf{(\mathfrak{B}\leftarrow A)\cdot \mathfrak{B}}="
        r"\mathbf{\lambda\;*\;\;I}"
        rf"\;?\;\rightarrow\;{np.isclose(t1, t2).all()}"
    )
)

---
**Efficient matrix exponentiation via eigendecomposition.**
Computing $\mathbf{A}^n$ directly requires $n-1$ matrix multiplications.
Using eigendecomposition $\mathbf{A} = \mathfrak{B}\,\mathbf{\Lambda}\,\mathfrak{B}^{-1}$:

$$\mathbf{A}^n = \mathfrak{B}\,\mathbf{\Lambda}^n\,\mathfrak{B}^{-1}$$

Since $\mathbf{\Lambda}$ is diagonal, $\mathbf{\Lambda}^n$ simply raises each
diagonal entry to the $n$-th power - a trivial $O(d)$ operation for a
$d\times d$ matrix.
Here $\mathbf{A}^{100}$ is computed both ways and confirmed equal.

In [ ]:
# Cell 06 - A^100 via direct power vs eigendecomposition: B * Lambda^100 * B^-1

a = np.array([[0.8, 0.3], [0.2, 0.7]])
display(as_latex(a, prefix=r"\mathbf{A}="))

eigen_vals, eigen_vecs = np.linalg.eig(a)
b = eigen_vecs.copy()
display(as_latex(b, prefix=r"\mathrm{Eigenbasis\;(\mathfrak{B})\;of\;}\mathbf{A}="))

lam = eigen_vals[np.newaxis] * np.identity(2)  # diagonal eigenvalue matrix
display(as_latex(lam, prefix=r"\mathbf{\color{#66FF66}{\Lambda}}="))

pow_lam = np.linalg.matrix_power(lam, 100)  # raise diagonal entries to the 100th
display(as_latex(pow_lam, prefix=r"\mathbf{\color{#66FF66}{\Lambda^{100}}}="))

t1 = np.linalg.matrix_power(a, 100)  # direct: 99 multiplications
t2 = np.dot(b, np.dot(pow_lam, np.linalg.inv(b)))  # via eigendecomposition

display(as_latex(t1, prefix=r"t_1=\mathbf{A^{100}}="))
display(
    as_latex(
        t2,
        prefix=(
            r"t_2="
            r"\mathbf{\mathfrak{B}\cdot\color{#66FF66}{\Lambda^{100}}}"
            r"\color{defaultcolor}{\;\cdot\;\mathfrak{B^{-1}}}"
        ),
    )
)
display(
    Math(
        r"\mathbf{A^{100}}="
        r"\mathbf{\mathfrak{B}\cdot\color{#66FF66}{\Lambda^{100}}}"
        r"\color{defaultcolor}{\;\cdot\;\mathfrak{B^{-1}}}"
        rf"\;?\;\rightarrow\;{np.isclose(t1, t2).all()}"
    )
)

---
**Matrices with a shared eigenbasis commute.**
In general, $\mathbf{A}\cdot\mathbf{B} \neq \mathbf{B}\cdot\mathbf{A}$.
However, if both matrices share the same eigenbasis $\mathfrak{B}$, then:

$$\mathbf{A}\cdot\mathbf{B}
= (\mathfrak{B}\mathbf{\Lambda}_A\mathfrak{B}^{-1})(\mathfrak{B}\mathbf{\Lambda}_B\mathfrak{B}^{-1})
= \mathfrak{B}\mathbf{\Lambda}_A\mathbf{\Lambda}_B\mathfrak{B}^{-1}
= \mathfrak{B}\mathbf{\Lambda}_B\mathbf{\Lambda}_A\mathfrak{B}^{-1}
= \mathbf{B}\cdot\mathbf{A}$$

since diagonal matrices always commute.
Here $\mathbf{A}$ and $\mathbf{B}$ are constructed from the same eigenbasis
but different eigenvalues, then verified to commute.

In [ ]:
# Cell 07 - Matrices built from the same eigenbasis commute: A*B = B*A

# Define four linearly independent eigenvectors
eigen_vec1 = np.array([-2.7 + 5.1j, 7.8 + 2j, -11, 9j])
eigen_vec2 = np.array([1.1j, -6.2 + 0.3j, 2j, -0.7])
eigen_vec3 = np.array([8, -2.4j, 1 + 1j, -4.2 - 8.1j])
eigen_vec4 = np.array([10 - 1j, 5.5, 9.1 + 1.9j, 0])

eigen_basis = np.array([eigen_vec1, eigen_vec2, eigen_vec3, eigen_vec4]).T
display(as_latex(eigen_basis, prefix=r"\mathrm{Eigenbasis\;\mathfrak{B}}="))

# Distinct eigenvalues for A and B - same eigenbasis, different scaling
eigen_valsA = np.array([2.9 - 4.3j, 4 + 0.9j, -7j, 3.9 - 10.2j])
eigen_valsB = np.array([5.7 - 2.3j, -4.1 + 2j, 8 - 7.1j, 0.3])

display(
    as_latex(eigen_valsA, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;}\mathbf{A}=")
)
display(
    as_latex(eigen_valsB, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;}\mathbf{B}=")
)

# Construct A and B from the shared eigenbasis via eigendecomposition
lamA = np.diag(eigen_valsA)
lamB = np.diag(eigen_valsB)

a = np.dot(eigen_basis, np.dot(lamA, np.linalg.inv(eigen_basis)))
b = np.dot(eigen_basis, np.dot(lamB, np.linalg.inv(eigen_basis)))

display(as_latex(a, prefix=r"\mathbf{A}="))
display(as_latex(b, prefix=r"\mathbf{B}="))

t1 = np.dot(a, b)
t2 = np.dot(b, a)

display(
    Math(
        r"\mathbf{A\cdot B}="
        r"\mathbf{B\cdot A}"
        rf"\;?\;\rightarrow\;{np.isclose(t1, t2).all()}"
    )
)

---
**Unitary matrices: eigenvalues and determinant lie on the unit circle.**
For a unitary matrix $\mathbf{U}$ ($\mathbf{U}\mathbf{U}^\dagger = \mathbf{I}$),
every eigenvalue $\lambda$ satisfies $|\lambda| = 1$.
This follows because $\|\mathbf{U}\mathbf{v}\| = \|\mathbf{v}\|$ and
$\mathbf{U}\mathbf{v} = \lambda\mathbf{v}$ imply $|\lambda|\|\mathbf{v}\| = \|\mathbf{v}\|$.
Since the determinant equals the product of eigenvalues,
$|\det(\mathbf{U})| = \prod_k |\lambda_k| = 1$ as well.
A random unitary matrix is generated via QR decomposition to confirm both properties.

In [ ]:
# Cell 08 - Eigenvalues and determinant of a unitary matrix all have modulus 1
# QR decomposition of a random complex matrix produces a unitary factor Q

n = 4
a = np.random.randn(n, n) + 1j * np.random.randn(n, n)
u, _ = np.linalg.qr(a)  # u is unitary

display(as_latex(u, prefix=r"\mathbf{U}="))

# Verify unitarity: U† * U = I
t1 = np.dot(u.conj().T, u)
t2 = np.identity(n)
display(
    Math(rf"\mathbf{{U}}\;\mathrm{{unitary}}?\;\rightarrow\;{np.isclose(t1, t2).all()}")
)

# Determinant: |det(U)| should equal 1
det = np.array([np.linalg.det(u)])
display(as_latex(det, prefix=r"\mathbf{det(U)}="))
display(as_latex(np.abs(det), prefix=r"\mathbf{|det(U)|}="))

# Eigenvalues: each |lambda| should equal 1
eigen_vals = np.linalg.eigvals(u)
display(
    as_latex(eigen_vals, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;}\mathbf{U}=")
)
display(
    as_latex(
        np.abs(eigen_vals),
        prefix=r"\mathrm{|Eigenvalues\;(\lambda)|\;of\;}\mathbf{U}=",
    )
)